# 05 — Stain normalization

Train one normalizer on reference patches, then apply it to other images so
their stain colors match. Needs the `stain` extra (TIAToolbox).

| Method | Strength |
|---|---|
| Reinhard | fast color-statistics matching; a good baseline |
| Macenko | optical-density stain separation; common for H&E |
| Vahadane | sparse stain separation; often preserves structure best |

Fit on the training split only and reuse the weights for validation and test
data.

In [ ]:
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository whether Jupyter starts at its root or in how_to_use/."""
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "rocqipath").is_dir():
            return candidate
    return here


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"          # your slides (kept out of git)
RESULTS_ROOT = PROJECT_ROOT / "results"    # outputs (kept out of git)
DEMO_ROOT = PROJECT_ROOT / "notebook_demo_outputs"  # synthetic examples

import rocqipath as rp

print(f"RocqiPath {rp.__version__}")
print(f"Data    : {DATA_ROOT}")
print(f"Results : {RESULTS_ROOT}")

In [ ]:
PATCHES = RESULTS_ROOT / "patches"   # output of notebook 04, or any image folder
OUTPUT_DIR = RESULTS_ROOT / "stain"

METHOD = "macenko"                   # "reinhard", "macenko" or "vahadane"
STAINS = ["he"]                      # which patches to use; ["all"] for every image

RUN_TRAIN = False
RUN_APPLY = False

In [ ]:
if RUN_TRAIN:
    weights = rp.train_stain_normalizer(PATCHES, OUTPUT_DIR, method=METHOD, stains=STAINS,
                                        fit_min_tissue=0.10, max_train_patches=500)
    print("Weights:", weights.by_role("weights")[0].path)
else:
    weights = OUTPUT_DIR / "stain_normalization" / f"{METHOD}_weights.npz"
    print("Set RUN_TRAIN=True to fit. Expected weights:", weights)

`normalizer=` accepts the training result or a saved `.npz`; the method is
read from the file name. `resume=True` skips images already written.

In [ ]:
if RUN_APPLY:
    normalized = rp.normalize_stain(PATCHES, OUTPUT_DIR, normalizer=weights, stains=STAINS, resume=True)
    print(normalized.summary)
else:
    normalized = None
    print("Set RUN_APPLY=True once weights exist.")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

if normalized is not None and len(normalized):
    after = normalized.items[0].path
    before = next(p for p in PATCHES.rglob(after.name))
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    for ax, path, title in zip(axes, (before, after), ("Original", "Normalized")):
        ax.imshow(Image.open(path).convert("RGB"))
        ax.set_title(title)
        ax.axis("off")
    plt.show()

**Frequent issues** — no patches found: `stains` matches folder names or the
recorded patch stain, use `["all"]` otherwise; no tissue passed: inspect the
patches before lowering `fit_min_tissue`; color casts: improve the training
cohort rather than tuning to one example.